# Automated Facial Affect Recognition for PTSD Subtype Classification

**Experimental research pipeline** — *not* a diagnostic tool.

## Study context

This notebook implements the feature-extraction → modelling → evaluation →
explainability pipeline for a psychology paper that aims to distinguish:

- **Classic PTSD**
- **Dissociative PTSD (D-PTSD)**
- **Healthy Controls**

…using four complementary facial signals: **action units (AUs)**, **region-weighted
facial scores**, **head-pose dynamics**, and **temporal behavioural features**,
fused in a late-fusion multi-input classifier.

## ⚠️ Clinical & ethical notice

- This is an **experimental research model**, **not** a clinical diagnostic or
  treatment tool.
- **No public affect dataset contains PTSD diagnoses.** The six emotion/AU states
  come from public datasets; the `Classic PTSD / D-PTSD / Control` labels must be
  **study-provided** or applied as an **explicitly documented proxy mapping**
  (see the Manifest).
- Only **public or request-authorized** data is used; **no private patient
  identifiers** are processed or stored.

## How to run

1. **Runtime → Change runtime type → T4 GPU** (free) or **A100** (Pro).
2. Run **Requirements**, then **Smoke test** (tiny synthetic end-to-end run).
3. Proceed **Data → Features → Model → Train → Evaluate → Explain**.
4. Checkpoints are saved to Google Drive (`MyDrive/ptsd_affect/checkpoints/`) and
   training **auto-resumes** after a session timeout.

## Sections

| Section | Purpose |
|---|---|
| Data | Acquire CK+ / AffectNet / RAF-DB (and request DISFA / BP4D) |
| Features | 28-dim frame features: AU + region + pose + temporal |
| Model | Multi-input late-fusion MLP with 6 + 3 output heads |
| Train | AMP + gradient checkpointing + early stopping + Drive checkpoints |
| Evaluate | Metrics, confusion matrix, ROC, subtype report, paper tables |
| Explain | SHAP feature importance + Grad-CAM region overlays |


In [ ]:
# ============================================================
# Requirements  (run once)  +  fixed random seed
# ============================================================
import sys, os
print('Python', sys.version)

# Base numerical / ML stack
!pip install -q numpy pandas scikit-learn matplotlib seaborn pillow

# Deep learning (T4 / A100)
!pip install -q torch torchvision

# Facial feature extraction (choose one; py-feat is easiest on Colab)
!pip install -q py-feat opencv-python-headless

# Explainability
!pip install -q shap

# Dataset acquisition
!pip install -q gdown kagglehub kaggle

# Mount Google Drive for checkpoint persistence
from google.colab import drive
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/ptsd_affect/checkpoints', exist_ok=True)
print('Drive mounted.')


In [ ]:
# Write the modular pipeline source to disk and import it.
# (This mirrors the `src/*.py` files exactly, so the notebook is a
#  single self-contained file but keeps clean module boundaries.)
import os, sys, json
PIPE = '/content/pipeline'
os.makedirs(PIPE, exist_ok=True)
if PIPE not in sys.path:
    sys.path.insert(0, PIPE)

SRC = {}
SRC['config.py'] = '"""\nconfig.py — Central configuration for the PTSD facial-affect recognition pipeline.\n\nEverything here is imported by the other modules and by the integrated Colab\nnotebook so that constants (AU list, class labels, feature dimensions, seed)\nlive in exactly one place.\n"""\nimport os\nimport random\n\n# ---------------------------------------------------------------------------\n# Reproducibility\n# ---------------------------------------------------------------------------\nSEED = 42\n\n\ndef seed_everything(seed: int = SEED) -> None:\n    """Fix Python, NumPy and PyTorch seeds for reproducible runs."""\n    import numpy as np\n    random.seed(seed)\n    os.environ["PYTHONHASHSEED"] = str(seed)\n    np.random.seed(seed)\n    try:\n        import torch\n        torch.manual_seed(seed)\n        torch.cuda.manual_seed_all(seed)\n        torch.backends.cudnn.deterministic = True\n        torch.backends.cudnn.benchmark = False\n    except ImportError:\n        pass\n\n\n# ---------------------------------------------------------------------------\n# Label sets\n# ---------------------------------------------------------------------------\n# Six emotion / affect states (primary classification head)\nEMOTION_CLASSES = ["Fear", "Anger", "Sadness", "Neutral", "Surprise", "Flat/Blunted Affect"]\nN_EMOTION = len(EMOTION_CLASSES)\n\n# Secondary head: PTSD subtype / group\nSUBTYPE_CLASSES = ["Classic PTSD", "D-PTSD", "Control"]\nN_SUBTYPE = len(SUBTYPE_CLASSES)\n\n# ---------------------------------------------------------------------------\n# Action Unit schema (canonical 17-AU set produced by OpenFace / py-feat)\n# ---------------------------------------------------------------------------\nAU_NAMES = [\n    "AU01", "AU02", "AU04", "AU05", "AU06", "AU07", "AU09", "AU10",\n    "AU12", "AU14", "AU15", "AU17", "AU20", "AU23", "AU25", "AU26", "AU45",\n]\nN_AU = len(AU_NAMES)\n\n# Region definitions (indices into AU_NAMES). Four regions as requested:\n# eyes / mouth (fine diagnosticity) and upper-face / lower-face (broad).\nREGION_AU_MAP = {\n    "eye":        ["AU01", "AU02", "AU04", "AU05", "AU06", "AU07", "AU45"],\n    "mouth":      ["AU09", "AU10", "AU12", "AU14", "AU15", "AU17", "AU20", "AU23", "AU25", "AU26"],\n    "upper_face": ["AU01", "AU02", "AU04", "AU05", "AU06", "AU07"],\n    "lower_face": ["AU09", "AU10", "AU12", "AU14", "AU15", "AU17", "AU20", "AU23", "AU25", "AU26"],\n}\nREGION_NAMES = list(REGION_AU_MAP.keys())          # eye, mouth, upper_face, lower_face\nN_REGION = len(REGION_NAMES)\n\n# Diagnosticity weights for region scores. These are *literature-informed\n# priors* for PTSD / blunted-affect work and are fully tunable: brow lowering\n# (AU04), lid raising (AU05), cheek raising (AU06) and lip-corner pulling\n# (AU12) carry the most signal for reduced / dysregulated expressivity.\n# Defaults are intentionally transparent and overridable in the notebook.\nREGION_DIAGNOSTIC_WEIGHTS = {\n    "AU01": 1.0, "AU02": 1.0, "AU04": 1.5, "AU05": 1.5, "AU06": 1.5,\n    "AU07": 1.2, "AU09": 1.0, "AU10": 1.0, "AU12": 1.5, "AU14": 1.0,\n    "AU15": 1.0, "AU17": 1.0, "AU20": 1.1, "AU23": 1.0, "AU25": 1.0,\n    "AU26": 1.0, "AU45": 1.0,\n}\n\n# ---------------------------------------------------------------------------\n# Head pose + temporal feature names\n# ---------------------------------------------------------------------------\nPOSE_NAMES = ["yaw", "pitch", "roll"]\nN_POSE = len(POSE_NAMES)\n\nTEMPORAL_NAMES = ["dwell_time", "transition_rate", "entropy", "au_variability"]\nN_TEMPORAL = len(TEMPORAL_NAMES)\n\n# Full flat feature vector (28 dims) = AU(17) + region(4) + pose(3) + temporal(4)\nFEATURE_NAMES = (\n    [f"au_{a}" for a in AU_NAMES]\n    + [f"region_{r}" for r in REGION_NAMES]\n    + [f"pose_{p}" for p in POSE_NAMES]\n    + [f"temp_{t}" for t in TEMPORAL_NAMES]\n)\nN_FEATURES = N_AU + N_REGION + N_POSE + N_TEMPORAL\n\n# Branch dimensions fed as *separate* model inputs\nINPUT_DIMS = {\n    "au": N_AU,\n    "region": N_REGION,\n    "pose": N_POSE,\n    "temporal": N_TEMPORAL,\n}\n\n# ---------------------------------------------------------------------------\n# Paths (Colab defaults; override at runtime)\n# ---------------------------------------------------------------------------\nDATA_ROOT = "/content/data"\nDRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/ptsd_affect/checkpoints"\nLOCAL_CHECKPOINT_DIR = "/content/checkpoints"\nFIG_DIR = "/content/figures"\n\n# Temporal sliding window (frames) for sequence-level features\nTEMPORAL_WINDOW = 15\n\n# AU activation threshold for binarised dwell / entropy / transition features\nAU_ACTIVATION_THRESHOLD = 1.0\n\n# Emotion classes that are "confusion pairs" to highlight in the paper figure\nCONFUSION_PAIRS = [("Fear", "Surprise"), ("Anger", "Disgust")]\n'
SRC['features.py'] = '"""\nfeatures.py — Frame-level feature extraction.\n\nProduces, per face frame, a structured DataFrame with:\n  1. Action Unit (AU) intensities            (17 canonical AUs)\n  2. Region-weighted facial scores           (eye, mouth, upper_face, lower_face)\n  3. Head-pose angles                        (yaw, pitch, roll)\n  4. Temporal sequence features              (dwell time, transition rate,\n                                               entropy, AU variability)\n\nThe `build_feature_frame` and `add_temporal_features` functions are pure\nNumPy/Pandas and are the part of the pipeline covered by the local CPU smoke\ntest. The AU + pose values are assumed to come from OpenFace / py-feat; a\nsynthetic generator is provided for end-to-end demonstration and the smoke test.\n"""\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom config import (\n    AU_NAMES, REGION_AU_MAP, REGION_NAMES, POSE_NAMES,\n    TEMPORAL_NAMES, FEATURE_NAMES, REGION_DIAGNOSTIC_WEIGHTS,\n    TEMPORAL_WINDOW, AU_ACTIVATION_THRESHOLD, N_FEATURES,\n)\n\n\n# ---------------------------------------------------------------------------\n# 1. Action Unit intensities\n# ---------------------------------------------------------------------------\ndef validate_au_matrix(au: np.ndarray) -> np.ndarray:\n    """Normalise and clip an (n_frames, N_AU) AU-intensity matrix."""\n    au = np.asarray(au, dtype=np.float32)\n    if au.ndim == 1:\n        au = au[None, :]\n    assert au.shape[1] == len(AU_NAMES), f"Expected {len(AU_NAMES)} AUs, got {au.shape[1]}"\n    au = np.clip(au, 0.0, 5.0)  # AU intensity is conventionally 0-5\n    return au\n\n\n# ---------------------------------------------------------------------------\n# 2. Region-weighted facial scores\n# ---------------------------------------------------------------------------\ndef region_scores(au: np.ndarray) -> np.ndarray:\n    """Region-weighted scores for eye/mouth/upper_face/lower_face.\n\n    Each region score is a diagnosticity-weighted mean of its member AUs:\n        score = sum(w_i * au_i) / sum(w_i)\n    Returns (n_frames, N_REGION).\n    """\n    au = validate_au_matrix(au)\n    out = np.zeros((au.shape[0], len(REGION_NAMES)), dtype=np.float32)\n    for j, region in enumerate(REGION_NAMES):\n        members = REGION_AU_MAP[region]\n        idx = [AU_NAMES.index(a) for a in members]\n        w = np.array([REGION_DIAGNOSTIC_WEIGHTS[a] for a in members], dtype=np.float32)\n        out[:, j] = (au[:, idx] * w).sum(axis=1) / w.sum()\n    return out\n\n\n# ---------------------------------------------------------------------------\n# 3. Head-pose angles\n# ---------------------------------------------------------------------------\ndef validate_pose(pose: np.ndarray) -> np.ndarray:\n    """Normalise an (n_frames, 3) pose matrix of [yaw, pitch, roll] in degrees."""\n    pose = np.asarray(pose, dtype=np.float32)\n    if pose.ndim == 1:\n        pose = pose[None, :]\n    assert pose.shape[1] == len(POSE_NAMES), f"Expected {len(POSE_NAMES)} pose angles"\n    return pose\n\n\n# ---------------------------------------------------------------------------\n# 4. Temporal sequence features (causal sliding window)\n# ---------------------------------------------------------------------------\ndef _binary_active(au: np.ndarray, thr: float = AU_ACTIVATION_THRESHOLD) -> np.ndarray:\n    """Binarise AU activations above threshold."""\n    return (au >= thr).astype(np.int8)\n\n\ndef dwell_time(au: np.ndarray, window: int = TEMPORAL_WINDOW,\n               thr: float = AU_ACTIVATION_THRESHOLD) -> np.ndarray:\n    """Fraction of frames in the trailing window where >=1 diagnostic AU is active.\n\n    Captures how long a face \'holds\' an expression (blunted affect -> low dwell).\n    """\n    act = _binary_active(au, thr)\n    any_active = (act.sum(axis=1) > 0).astype(np.float32)\n    return _rolling_mean(any_active, window)\n\n\ndef transition_rate(au: np.ndarray, window: int = TEMPORAL_WINDOW,\n                    thr: float = AU_ACTIVATION_THRESHOLD) -> np.ndarray:\n    """Mean per-frame count of AU on/off transitions in the trailing window.\n\n    A transition is any AU changing active-state between consecutive frames.\n    """\n    act = _binary_active(au, thr)\n    # per-frame number of AUs that flipped state\n    flips = np.abs(np.diff(act, axis=0)).sum(axis=1).astype(np.float32)\n    flips = np.concatenate([np.zeros(1, dtype=np.float32), flips])\n    return _rolling_mean(flips, window)\n\n\ndef au_entropy(au: np.ndarray, window: int = TEMPORAL_WINDOW,\n               thr: float = AU_ACTIVATION_THRESHOLD) -> np.ndarray:\n    """Shannon entropy of the binarised AU activation vector (per frame),\n    averaged over the trailing window. High entropy -> more complex / mixed\n    facial behaviour; low entropy -> restricted, flat repertoire.\n    """\n    act = _binary_active(au, thr)\n    # entropy over the N_AU binary channels per frame\n    p_active = act.mean(axis=1)\n    p_active = np.clip(p_active, 1e-9, 1.0 - 1e-9)\n    ent = -(p_active * np.log2(p_active) + (1 - p_active) * np.log2(1 - p_active))\n    return _rolling_mean(ent.astype(np.float32), window)\n\n\ndef au_variability(au: np.ndarray, window: int = TEMPORAL_WINDOW) -> np.ndarray:\n    """Mean per-AU standard deviation over the trailing window."""\n    out = np.zeros(au.shape[0], dtype=np.float32)\n    for t in range(au.shape[0]):\n        lo = max(0, t - window + 1)\n        out[t] = au[lo:t + 1].std(axis=0).mean()\n    return out\n\n\ndef _rolling_mean(x: np.ndarray, window: int) -> np.ndarray:\n    """Causal rolling mean (uses only past + current frames)."""\n    x = np.asarray(x, dtype=np.float32)\n    out = np.empty_like(x)\n    running = 0.0\n    for t in range(x.shape[0]):\n        running += x[t]\n        if t >= window:\n            running -= x[t - window]\n        out[t] = running / min(t + 1, window)\n    return out\n\n\ndef temporal_features(au: np.ndarray, window: int = TEMPORAL_WINDOW) -> np.ndarray:\n    """Stack the four temporal features into an (n_frames, N_TEMPORAL) matrix."""\n    return np.column_stack([\n        dwell_time(au, window),\n        transition_rate(au, window),\n        au_entropy(au, window),\n        au_variability(au, window),\n    ]).astype(np.float32)\n\n\n# ---------------------------------------------------------------------------\n# Full frame-level DataFrame\n# ---------------------------------------------------------------------------\ndef build_feature_frame(au: np.ndarray, pose: np.ndarray,\n                        sequence_ids: np.ndarray | list[str] | None = None,\n                        frame_idx: np.ndarray | None = None,\n                        emotion_labels: list[str] | np.ndarray | None = None,\n                        subtype_labels: list[str] | np.ndarray | None = None,\n                        window: int = TEMPORAL_WINDOW) -> pd.DataFrame:\n    """Assemble one row per frame with all AU / region / pose / temporal columns.\n\n    Temporal features are computed per-sequence (grouped by `sequence_ids`) so\n    that windows never bleed across sequence boundaries.\n    """\n    au = validate_au_matrix(au)\n    pose = validate_pose(pose)\n    n = au.shape[0]\n    assert pose.shape[0] == n, "AU and pose must have the same number of frames"\n\n    if sequence_ids is None:\n        sequence_ids = np.zeros(n, dtype=int)\n    sequence_ids = np.asarray(sequence_ids)\n\n    if frame_idx is None:\n        frame_idx = np.arange(n)\n    frame_idx = np.asarray(frame_idx)\n\n    regions = region_scores(au)\n\n    # temporal features per sequence\n    temporal = np.zeros((n, len(TEMPORAL_NAMES)), dtype=np.float32)\n    for sid in np.unique(sequence_ids):\n        mask = sequence_ids == sid\n        temporal[mask] = temporal_features(au[mask], window)\n\n    cols = {}\n    for j, a in enumerate(AU_NAMES):\n        cols[f"au_{a}"] = au[:, j]\n    for j, r in enumerate(REGION_NAMES):\n        cols[f"region_{r}"] = regions[:, j]\n    for j, p in enumerate(POSE_NAMES):\n        cols[f"pose_{p}"] = pose[:, j]\n    for j, t in enumerate(TEMPORAL_NAMES):\n        cols[f"temp_{t}"] = temporal[:, j]\n\n    df = pd.DataFrame(cols, columns=FEATURE_NAMES)\n    df.insert(0, "frame_idx", frame_idx)\n    df.insert(0, "sequence_id", sequence_ids)\n    if emotion_labels is not None:\n        df["emotion"] = list(emotion_labels)\n    if subtype_labels is not None:\n        df["subtype"] = list(subtype_labels)\n    return df\n\n\n# ---------------------------------------------------------------------------\n# Synthetic demo generator (for notebook demo + local smoke test)\n# ---------------------------------------------------------------------------\ndef generate_synthetic_dataset(n_sequences: int = 6, frames_per_seq: int = 60,\n                               n_au: int = None, seed: int = 0):\n    """Create plausible synthetic AU + pose + labels for end-to-end testing.\n\n    Returns (df, flat_X, y_emotion, y_subtype). Real runs replace this with\n    OpenFace/py-feat output. Emphasises that the pipeline can be exercised end\n    to end before any real (possibly request-gated) data arrives.\n    """\n    from config import EMOTION_CLASSES, SUBTYPE_CLASSES\n    rng = np.random.default_rng(seed)\n    n_au = n_au or len(AU_NAMES)\n    aus, poses, seq_ids, frames, emo, sub = [], [], [], [], [], []\n\n    # emotion -> characteristic AU \'activation pattern\' (mean intensity)\n    emo_au_prior = {\n        "Fear":     {"AU01": 2.5, "AU02": 2.5, "AU05": 3.0, "AU20": 2.0, "AU26": 2.5},\n        "Anger":    {"AU04": 3.0, "AU05": 2.0, "AU07": 2.5, "AU23": 2.0},\n        "Sadness":  {"AU01": 2.5, "AU04": 2.0, "AU15": 2.5, "AU17": 2.0},\n        "Neutral":  {},\n        "Surprise": {"AU01": 2.5, "AU02": 3.0, "AU05": 3.0, "AU25": 2.0, "AU26": 3.0},\n        "Flat/Blunted Affect": {},\n    }\n\n    for s in range(n_sequences):\n        e = EMOTION_CLASSES[s % len(EMOTION_CLASSES)]\n        prior = emo_au_prior[e]\n        seq_au = np.zeros((frames_per_seq, n_au), dtype=np.float32)\n        for j, a in enumerate(AU_NAMES[:n_au]):\n            base = prior.get(a, rng.uniform(0.0, 0.4))\n            seq_au[:, j] = np.clip(rng.normal(base, 0.5, frames_per_seq), 0, 5)\n        # pose: small smooth wander\n        t = np.arange(frames_per_seq)\n        pose = np.column_stack([\n            15 * np.sin(2 * np.pi * t / 60) + rng.normal(0, 2, frames_per_seq),\n            8 * np.cos(2 * np.pi * t / 45) + rng.normal(0, 2, frames_per_seq),\n            6 * np.sin(2 * np.pi * t / 30) + rng.normal(0, 1.5, frames_per_seq),\n        ])\n        aus.append(seq_au)\n        poses.append(pose)\n        seq_ids += [s] * frames_per_seq\n        frames += list(range(frames_per_seq))\n        emo += [e] * frames_per_seq\n        sub += [SUBTYPE_CLASSES[s % len(SUBTYPE_CLASSES)]] * frames_per_seq\n\n    au = np.concatenate(aus)\n    pose = np.concatenate(poses)\n    df = build_feature_frame(au, pose, sequence_ids=seq_ids, frame_idx=frames,\n                             emotion_labels=emo, subtype_labels=sub)\n    X = df[FEATURE_NAMES].to_numpy(dtype=np.float32)\n    return df, X, emo, sub\n\n\nif __name__ == "__main__":\n    df, X, y_emo, y_sub = generate_synthetic_dataset(seed=0)\n    print("Feature frame shape:", df.shape)\n    print("Feature vector dim :", X.shape[1], "(expected", N_FEATURES, ")")\n    print(df[["sequence_id", "frame_idx", "au_AU04", "region_mouth",\n              "pose_yaw", "temp_entropy", "emotion", "subtype"]].head(8).to_string(index=False))\n'
SRC['tables.py'] = '"""\ntables.py — Copy-paste-ready LaTeX + Markdown result tables for the manuscript.\n\nTwo tables:\n  1. Emotion classification  ->  Emotion Class | Precision | Recall | F1 | AUC\n  2. PTSD subtype classification ->  Metric | Value  (Accuracy, Macro-F1, Kappa)\n\nBoth are produced from plain Python dicts (no placeholders), so they can be\ngenerated directly from the evaluation module\'s outputs.\n"""\nfrom __future__ import annotations\n\n\ndef _fmt(v, nd: int = 3) -> str:\n    """Format a float, leaving integers / strings untouched."""\n    if isinstance(v, str):\n        return v\n    try:\n        f = float(v)\n    except (TypeError, ValueError):\n        return str(v)\n    return f"{f:.{nd}f}"\n\n\ndef emotion_table_markdown(rows: list[dict], classes: list[str] | None = None) -> str:\n    """`rows` is a list of dicts with keys: class, precision, recall, f1, auc."""\n    header = "| Emotion Class | Precision | Recall | F1 | AUC |\\n|---|---|---|---|---|"\n    lines = [header]\n    order = classes or [r["class"] for r in rows]\n    by_class = {r["class"]: r for r in rows}\n    for c in order:\n        r = by_class[c]\n        lines.append(\n            f"| {r[\'class\']} | {_fmt(r[\'precision\'])} | {_fmt(r[\'recall\'])} "\n            f"| {_fmt(r[\'f1\'])} | {_fmt(r[\'auc\'])} |"\n        )\n    return "\\n".join(lines)\n\n\ndef emotion_table_latex(rows: list[dict], classes: list[str] | None = None,\n                        caption: str = "Per-class emotion recognition performance.",\n                        label: str = "tab:emotion_metrics") -> str:\n    """Return a journal-style LaTeX `table` environment (booktabs)."""\n    order = classes or [r["class"] for r in rows]\n    by_class = {r["class"]: r for r in rows}\n    body = []\n    for c in order:\n        r = by_class[c]\n        body.append(\n            f"        {r[\'class\']} & {_fmt(r[\'precision\'])} & {_fmt(r[\'recall\'])} "\n            f"& {_fmt(r[\'f1\'])} & {_fmt(r[\'auc\'])} \\\\\\\\"\n        )\n    return "\\n".join([\n        "\\\\begin{table}[htbp]",\n        "    \\\\centering",\n        "    \\\\caption{" + caption + "}",\n        "    \\\\label{" + label + "}",\n        "    \\\\begin{tabular}{lcccc}",\n        "        \\\\toprule",\n        "        Emotion Class & Precision & Recall & F1 & AUC \\\\\\\\",\n        "        \\\\midrule",\n        *body,\n        "        \\\\bottomrule",\n        "    \\\\end{tabular}",\n        "\\\\end{table}",\n    ])\n\n\ndef subtype_table_markdown(metrics: dict) -> str:\n    """`metrics` = {accuracy, macro_f1, cohen_kappa, n_samples}."""\n    lines = [\n        "| Metric | Value |",\n        "|---|---|",\n        f"| Accuracy | {_fmt(metrics[\'accuracy\'])} |",\n        f"| Macro-F1 | {_fmt(metrics[\'macro_f1\'])} |",\n        f"| Cohen\'s Kappa | {_fmt(metrics[\'cohen_kappa\'])} |",\n        f"| N (samples) | {metrics.get(\'n_samples\', \'-\')} |",\n    ]\n    return "\\n".join(lines)\n\n\ndef subtype_table_latex(metrics: dict,\n                        caption: str = "PTSD subtype classification performance.",\n                        label: str = "tab:subtype_metrics") -> str:\n    """Return a journal-style LaTeX table for the 3-way subtype classifier."""\n    return "\\n".join([\n        "\\\\begin{table}[htbp]",\n        "    \\\\centering",\n        "    \\\\caption{" + caption + "}",\n        "    \\\\label{" + label + "}",\n        "    \\\\begin{tabular}{lc}",\n        "        \\\\toprule",\n        "        Metric & Value \\\\\\\\",\n        "        \\\\midrule",\n        f"        Accuracy & {_fmt(metrics[\'accuracy\'])} \\\\\\\\",\n        f"        Macro-F1 & {_fmt(metrics[\'macro_f1\'])} \\\\\\\\",\n        f"        Cohen\'s Kappa & {_fmt(metrics[\'cohen_kappa\'])} \\\\\\\\",\n        f"        N (samples) & {metrics.get(\'n_samples\', \'-\')} \\\\\\\\",\n        "        \\\\bottomrule",\n        "    \\\\end{tabular}",\n        "\\\\end{table}",\n    ])\n\n\ndef save_tables(emotion_rows: list[dict], subtype_metrics: dict,\n                out_dir: str, classes: list[str] | None = None) -> dict:\n    """Write both tables to .md and .tex files and return the written paths."""\n    import os\n    os.makedirs(out_dir, exist_ok=True)\n    paths = {}\n\n    emo_md = emotion_table_markdown(emotion_rows, classes)\n    emo_tex = emotion_table_latex(emotion_rows, classes)\n    sub_md = subtype_table_markdown(subtype_metrics)\n    sub_tex = subtype_table_latex(subtype_metrics)\n\n    paths["emotion_md"] = os.path.join(out_dir, "table_emotion_metrics.md")\n    paths["emotion_tex"] = os.path.join(out_dir, "table_emotion_metrics.tex")\n    paths["subtype_md"] = os.path.join(out_dir, "table_subtype_metrics.md")\n    paths["subtype_tex"] = os.path.join(out_dir, "table_subtype_metrics.tex")\n\n    with open(paths["emotion_md"], "w") as f:\n        f.write(emo_md + "\\n")\n    with open(paths["emotion_tex"], "w") as f:\n        f.write(emo_tex + "\\n")\n    with open(paths["subtype_md"], "w") as f:\n        f.write(sub_md + "\\n")\n    with open(paths["subtype_tex"], "w") as f:\n        f.write(sub_tex + "\\n")\n\n    paths["emotion_markdown"] = emo_md\n    paths["emotion_latex"] = emo_tex\n    paths["subtype_markdown"] = sub_md\n    paths["subtype_latex"] = sub_tex\n    return paths\n'
SRC['evaluate.py'] = '"""\nevaluate.py — Evaluation, confusion analysis, ROC curves and paper-ready tables.\n\nProduces:\n  1. Per-class precision / recall / F1 for the six emotion states.\n  2. Confusion-matrix heatmap with fear-surprise and anger-disgust pairs\n     visually highlighted.\n  3. PTSD subtype classification report (Classic PTSD / D-PTSD / Control).\n  4. One-vs-rest ROC curves per emotion class (macro/micro averages).\n  5. A summary table for direct paste into a psychology journal paper.\n\nPlots follow a Fathom Information Design "scientific journal" aesthetic:\nneutral grays + navy + one highlight colour, restrained gridlines, high DPI.\n"""\nfrom __future__ import annotations\n\nimport os\nimport numpy as np\nfrom sklearn.metrics import (\n    classification_report, confusion_matrix, roc_curve, auc,\n    precision_recall_fscore_support, cohen_kappa_score, accuracy_score,\n)\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\n\nfrom config import EMOTION_CLASSES, SUBTYPE_CLASSES, CONFUSION_PAIRS\n\n# Fathom-style palette: navy, mid-gray, one warm highlight, neutral face\nNAVY = "#1F3A5F"\nGRAY = "#8A919C"\nHIGHLIGHT = "#C75B39"\nLIGHT = "#ECEFF3"\nFACE = "#F7F6F3"\n\n\ndef _apply_style() -> None:\n    plt.rcParams.update({\n        "figure.facecolor": FACE,\n        "axes.facecolor": FACE,\n        "axes.edgecolor": "#444",\n        "axes.grid": True,\n        "grid.color": LIGHT,\n        "grid.linewidth": 0.6,\n        "font.family": "DejaVu Sans",\n        "font.size": 10,\n        "axes.titlesize": 12,\n        "axes.titleweight": "bold",\n        "axes.labelcolor": "#222",\n        "xtick.color": "#444",\n        "ytick.color": "#444",\n        "figure.dpi": 150,\n        "savefig.dpi": 300,\n        "savefig.bbox": "tight",\n    })\n\n\ndef per_class_metrics(y_true: list[str], y_pred: list[str],\n                      classes: list[str]) -> dict:\n    """Return per-class precision/recall/f1 + AUC placeholders (filled later)."""\n    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=classes,\n                                                 average=None, zero_division=0)\n    return {c: {"precision": float(p[i]), "recall": float(r[i]), "f1": float(f[i])}\n            for i, c in enumerate(classes)}\n\n\ndef attach_auc(metrics: dict, y_true: list[str], y_scores: np.ndarray,\n               classes: list[str]) -> dict:\n    """Attach one-vs-rest ROC AUC per class given predicted probabilities."""\n    y_true_bin = _one_hot(y_true, classes)\n    for i, c in enumerate(classes):\n        try:\n            fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_scores[:, i])\n            metrics[c]["auc"] = float(auc(fpr, tpr))\n            metrics[c]["fpr"] = fpr\n            metrics[c]["tpr"] = tpr\n        except ValueError:\n            metrics[c]["auc"] = float("nan")\n    return metrics\n\n\ndef _one_hot(y: list[str], classes: list[str]) -> np.ndarray:\n    idx = {c: i for i, c in enumerate(classes)}\n    out = np.zeros((len(y), len(classes)), dtype=np.int8)\n    for i, lab in enumerate(y):\n        out[i, idx[lab]] = 1\n    return out\n\n\n# ---------------------------------------------------------------------------\n# Confusion matrix\n# ---------------------------------------------------------------------------\ndef plot_confusion_matrix(y_true: list[str], y_pred: list[str],\n                          classes: list[str], out_path: str,\n                          highlight_pairs: list[tuple] | None = None,\n                          normalize: str = "true") -> str:\n    """Confusion matrix heatmap; highlights the given (true, pred) confusion\n    pairs with a warm outline + annotation. Returns the saved path."""\n    highlight_pairs = highlight_pairs if highlight_pairs is not None else CONFUSION_PAIRS\n    _apply_style()\n\n    cm = confusion_matrix(y_true, y_pred, labels=classes)\n    if normalize == "true":\n        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1e-9)\n    elif normalize == "pred":\n        cm_norm = cm.astype(float) / cm.sum(axis=0, keepdims=True).clip(min=1e-9)\n    else:\n        cm_norm = cm.astype(float)\n\n    fig, ax = plt.subplots(figsize=(7.2, 6.4))\n    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1, aspect="auto")\n\n    # highlight confusion pairs\n    idx = {c: i for i, c in enumerate(classes)}\n    for a, b in highlight_pairs:\n        if a in idx and b in idx:\n            for (ti, pi) in [(idx[a], idx[b]), (idx[b], idx[a])]:\n                ax.add_patch(plt.Rectangle((pi - 0.5, ti - 0.5), 1, 1,\n                                           fill=False, edgecolor=HIGHLIGHT,\n                                           linewidth=2.2, zorder=3))\n\n    ax.set_xticks(range(len(classes)))\n    ax.set_yticks(range(len(classes)))\n    ax.set_xticklabels(classes, rotation=45, ha="right")\n    ax.set_yticklabels(classes)\n    ax.set_xlabel("Predicted")\n    ax.set_ylabel("True")\n    ax.set_title("Emotion confusion matrix (row-normalised)")\n\n    # annotate counts + fraction\n    thresh = cm_norm.max() / 2.0\n    for i in range(len(classes)):\n        for j in range(len(classes)):\n            label = f"{cm[i, j]}\\n{cm_norm[i, j]:.2f}"\n            ax.text(j, i, label, ha="center", va="center",\n                    color="white" if cm_norm[i, j] > thresh else "#1a1a1a",\n                    fontsize=8)\n\n    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)\n    fig.tight_layout()\n    fig.savefig(out_path)\n    plt.close(fig)\n    return out_path\n\n\n# ---------------------------------------------------------------------------\n# ROC curves\n# ---------------------------------------------------------------------------\ndef plot_roc_curves(y_true: list[str], y_scores: np.ndarray,\n                    classes: list[str], out_path: str) -> str:\n    """One-vs-rest ROC curves + macro/micro average. Returns saved path."""\n    _apply_style()\n    y_bin = _one_hot(y_true, classes)\n    n = len(classes)\n    colors = [NAVY, GRAY, HIGHLIGHT, "#5B7A9D", "#A6B0BC", "#8C5A3B"]\n\n    fig, ax = plt.subplots(figsize=(6.6, 6.0))\n    tprs, aucs = [], []\n    mean_fpr = np.linspace(0, 1, 100)\n    for i, c in enumerate(classes):\n        fpr, tpr, _ = roc_curve(y_bin[:, i], y_scores[:, i])\n        a = auc(fpr, tpr)\n        aucs.append(a)\n        ax.plot(fpr, tpr, color=colors[i], lw=1.8,\n                label=f"{c} (AUC={a:.2f})")\n        interp = np.interp(mean_fpr, fpr, tpr)\n        interp[0] = 0.0\n        tprs.append(interp)\n\n    mean_tpr = np.mean(tprs, axis=0)\n    mean_tpr[-1] = 1.0\n    macro_auc = auc(mean_fpr, mean_tpr)\n    ax.plot(mean_fpr, mean_tpr, color=NAVY, linestyle="--", lw=2.2,\n            label=f"Macro-avg (AUC={macro_auc:.2f})")\n\n    ax.plot([0, 1], [0, 1], color="#bbb", linestyle=":", lw=1.2, label="Chance")\n    ax.set_xlim([0, 1]); ax.set_ylim([0, 1])\n    ax.set_xlabel("False positive rate")\n    ax.set_ylabel("True positive rate")\n    ax.set_title("One-vs-rest ROC curves (emotion states)")\n    ax.legend(loc="lower right", fontsize=8, frameon=False)\n    fig.tight_layout()\n    fig.savefig(out_path)\n    plt.close(fig)\n    return out_path\n\n\n# ---------------------------------------------------------------------------\n# Subtype report\n# ---------------------------------------------------------------------------\ndef subtype_report(y_true: list[str], y_pred: list[str],\n                   classes: list[str] | None = None) -> dict:\n    """Accuracy, macro-F1 and Cohen\'s kappa for the 3-way subtype task."""\n    classes = classes or SUBTYPE_CLASSES\n    y_true = list(y_true)\n    y_pred = list(y_pred)\n    acc = accuracy_score(y_true, y_pred)\n    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=classes,\n                                                 average="macro", zero_division=0)\n    kappa = cohen_kappa_score(y_true, y_pred, labels=classes)\n    report_str = classification_report(y_true, y_pred, labels=classes,\n                                       zero_division=0, digits=3)\n    return {\n        "accuracy": float(acc),\n        "macro_f1": float(f),\n        "cohen_kappa": float(kappa),\n        "n_samples": len(y_true),\n        "report": report_str,\n    }\n\n\n# ---------------------------------------------------------------------------\n# Convenience: full evaluation bundle\n# ---------------------------------------------------------------------------\ndef run_evaluation(y_emo_true, y_emo_pred, y_emo_score,\n                   y_sub_true, y_sub_pred, out_dir: str) -> dict:\n    """Run the whole evaluation and return a dict of metrics + figure paths."""\n    os.makedirs(out_dir, exist_ok=True)\n\n    emo_metrics = per_class_metrics(y_emo_true, y_emo_pred, EMOTION_CLASSES)\n    emo_metrics = attach_auc(emo_metrics, y_emo_true, np.asarray(y_emo_score),\n                             EMOTION_CLASSES)\n\n    cm_path = plot_confusion_matrix(y_emo_true, y_emo_pred, EMOTION_CLASSES,\n                                    os.path.join(out_dir, "confusion_matrix.png"))\n    roc_path = plot_roc_curves(y_emo_true, np.asarray(y_emo_score),\n                               EMOTION_CLASSES,\n                               os.path.join(out_dir, "roc_curves.png"))\n    sub = subtype_report(y_sub_true, y_sub_pred)\n\n    return {\n        "emotion_metrics": emo_metrics,\n        "subtype_metrics": sub,\n        "figures": {"confusion_matrix": cm_path, "roc_curves": roc_path},\n    }\n'
SRC['model.py'] = '"""\nmodel.py — Multi-input late-fusion classifier (PyTorch).\n\nAccepts FOUR separate inputs:\n  1. AU intensity vector        (17)\n  2. Region-weight vector       (4)\n  3. Head-pose vector           (3)\n  4. Temporal feature vector    (4)\n\nEach branch is a small MLP (Linear -> BatchNorm -> ReLU -> Dropout). Branch\nembeddings are concatenated (late fusion) and passed through a shared MLP,\nthen two heads predict:\n  - 6 emotion/affect states: Fear, Anger, Sadness, Neutral, Surprise, Flat/Blunted\n  - 3 PTSD subtypes: Classic PTSD, D-PTSD, Control\n\nDropout + BatchNorm are used throughout. An optional `use_gradient_checkpointing`\nflag wraps the shared MLP with torch.utils.checkpoint to reduce VRAM.\n"""\nfrom __future__ import annotations\n\nimport torch\nimport torch.nn as nn\n\nfrom config import (\n    N_AU, N_REGION, N_POSE, N_TEMPORAL, N_EMOTION, N_SUBTYPE,\n    EMOTION_CLASSES, SUBTYPE_CLASSES, FEATURE_NAMES,\n)\n\n\nclass BranchMLP(nn.Module):\n    """Single-input branch encoder."""\n\n    def __init__(self, in_dim: int, hidden: list[int], dropout: float = 0.3):\n        super().__init__()\n        layers = []\n        prev = in_dim\n        for h in hidden:\n            layers.append(nn.Linear(prev, h))\n            layers.append(nn.BatchNorm1d(h))\n            layers.append(nn.ReLU(inplace=True))\n            layers.append(nn.Dropout(dropout))\n            prev = h\n        self.net = nn.Sequential(*layers)\n        self.out_dim = prev\n\n    def forward(self, x):\n        return self.net(x)\n\n\nclass FusionMLP(nn.Module):\n    """Shared late-fusion MLP with optional gradient checkpointing."""\n\n    def __init__(self, in_dim: int, hidden: list[int], dropout: float = 0.35,\n                 use_checkpoint: bool = False):\n        super().__init__()\n        self.use_checkpoint = use_checkpoint\n        self.blocks = nn.ModuleList()\n        prev = in_dim\n        for h in hidden:\n            self.blocks.append(nn.Sequential(\n                nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(inplace=True),\n                nn.Dropout(dropout),\n            ))\n            prev = h\n        self.out_dim = prev\n\n    def forward(self, x):\n        for block in self.blocks:\n            if self.use_checkpoint and self.training and x.requires_grad:\n                x = torch.utils.checkpoint.checkpoint(block, x,\n                                                      use_reentrant=False)\n            else:\n                x = block(x)\n        return x\n\n\nclass MultiInputPTSDAffectModel(nn.Module):\n    def __init__(\n        self,\n        au_dim: int = N_AU,\n        region_dim: int = N_REGION,\n        pose_dim: int = N_POSE,\n        temporal_dim: int = N_TEMPORAL,\n        n_emotion: int = N_EMOTION,\n        n_subtype: int = N_SUBTYPE,\n        branch_hidden: list[int] | None = None,\n        fusion_hidden: list[int] | None = None,\n        dropout: float = 0.35,\n        use_gradient_checkpointing: bool = False,\n    ):\n        super().__init__()\n        branch_hidden = branch_hidden or [32]\n        fusion_hidden = fusion_hidden or [128, 64]\n\n        self.au_branch = BranchMLP(au_dim, branch_hidden, dropout)\n        self.region_branch = BranchMLP(region_dim, branch_hidden, dropout)\n        self.pose_branch = BranchMLP(pose_dim, branch_hidden, dropout)\n        self.temporal_branch = BranchMLP(temporal_dim, branch_hidden, dropout)\n\n        fusion_in = (self.au_branch.out_dim + self.region_branch.out_dim\n                     + self.pose_branch.out_dim + self.temporal_branch.out_dim)\n        self.fusion = FusionMLP(fusion_in, fusion_hidden, dropout,\n                                use_gradient_checkpointing)\n\n        self.emotion_head = nn.Linear(self.fusion.out_dim, n_emotion)\n        self.subtype_head = nn.Linear(self.fusion.out_dim, n_subtype)\n\n        self.n_emotion = n_emotion\n        self.n_subtype = n_subtype\n\n    def forward(self, au, region, pose, temporal):\n        a = self.au_branch(au)\n        r = self.region_branch(region)\n        p = self.pose_branch(pose)\n        t = self.temporal_branch(temporal)\n        fused = torch.cat([a, r, p, t], dim=1)\n        z = self.fusion(fused)\n        return self.emotion_head(z), self.subtype_head(z)\n\n    def forward_flat(self, x_flat):\n        """Convenience for SHAP: split a flat (N, 28) vector into the 4 branches."""\n        au = x_flat[:, :N_AU]\n        region = x_flat[:, N_AU:N_AU + N_REGION]\n        pose = x_flat[:, N_AU + N_REGION:N_AU + N_REGION + N_POSE]\n        temporal = x_flat[:, N_AU + N_REGION + N_POSE:]\n        return self.forward(au, region, pose, temporal)\n\n    def summary(self) -> str:\n        """Return a human-readable parameter summary table."""\n        total = sum(p.numel() for p in self.parameters())\n        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)\n        lines = [\n            "=" * 62,\n            "MultiInputPTSDAffectModel",\n            "=" * 62,\n            f"{\'Input branch\':<24}{\'in_dim\':>10}{\'out_dim\':>10}",\n            f"{\'AU\':<24}{N_AU:>10}{self.au_branch.out_dim:>10}",\n            f"{\'Region\':<24}{N_REGION:>10}{self.region_branch.out_dim:>10}",\n            f"{\'Head-pose\':<24}{N_POSE:>10}{self.pose_branch.out_dim:>10}",\n            f"{\'Temporal\':<24}{N_TEMPORAL:>10}{self.temporal_branch.out_dim:>10}",\n            "-" * 62,\n            f"Fusion input dim : {self.au_branch.out_dim + self.region_branch.out_dim + self.pose_branch.out_dim + self.temporal_branch.out_dim}",\n            f"Fusion output dim: {self.fusion.out_dim}",\n            f"Emotion head     : {self.n_emotion} classes {EMOTION_CLASSES}",\n            f"Subtype head     : {self.n_subtype} classes {SUBTYPE_CLASSES}",\n            "-" * 62,\n            f"Total params     : {total:,}",\n            f"Trainable params : {trainable:,}",\n            "=" * 62,\n        ]\n        return "\\n".join(lines)\n\n\n# ---------------------------------------------------------------------------\n# Lightweight CNN backbone for Grad-CAM (visualising attended facial regions)\n# ---------------------------------------------------------------------------\nclass GradCAMBackbone(nn.Module):\n    """Small CNN whose final conv feature map feeds Grad-CAM.\n\n    Used only for the *visualisation* of which facial regions the auxiliary\n    image pathway attends to. The primary model remains the feature-fusion MLP.\n    """\n\n    def __init__(self, in_channels: int = 3, n_emotion: int = N_EMOTION):\n        super().__init__()\n        self.features = nn.Sequential(\n            nn.Conv2d(in_channels, 32, 3, padding=1), nn.BatchNorm2d(32),\n            nn.ReLU(inplace=True), nn.MaxPool2d(2),\n            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),\n            nn.ReLU(inplace=True), nn.MaxPool2d(2),\n            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128),\n            nn.ReLU(inplace=True), nn.MaxPool2d(2),\n            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128),\n            nn.ReLU(inplace=True),\n        )\n        self.pool = nn.AdaptiveAvgPool2d((1, 1))\n        self.classifier = nn.Linear(128, n_emotion)\n        self.gradients = None\n        self.activations = None\n        self._hook()\n\n    def _hook(self):\n        def _forward(module, inp, out):\n            self.activations = out\n\n        def _backward(module, grad_in, grad_out):\n            self.gradients = grad_out[0]\n\n        self.features.register_forward_hook(_forward)\n        self.features.register_full_backward_hook(_backward)\n\n    def forward(self, x):\n        f = self.features(x)\n        pooled = self.pool(f).flatten(1)\n        return self.classifier(pooled)\n\n\ndef build_model(**kwargs) -> MultiInputPTSDAffectModel:\n    """Factory: build the multi-input model with sensible defaults."""\n    return MultiInputPTSDAffectModel(**kwargs)\n\n\nif __name__ == "__main__":\n    from config import seed_everything\n    seed_everything(0)\n    model = build_model()\n    print(model.summary())\n    b = 8\n    emo, sub = model(\n        torch.randn(b, N_AU), torch.randn(b, N_REGION),\n        torch.randn(b, N_POSE), torch.randn(b, N_TEMPORAL),\n    )\n    print("emotion logits:", tuple(emo.shape), "subtype logits:", tuple(sub.shape))\n'
SRC['train.py'] = '"""\ntrain.py — Colab-optimised training loop.\n\nFeatures:\n  - Mixed precision (torch.cuda.amp GradScaler + autocast)\n  - Gradient checkpointing toggle on the fusion MLP (saves VRAM)\n  - Early stopping (patience = 10) on validation loss\n  - Class-weighted cross-entropy for both heads (handles imbalance)\n  - Checkpoint saving to Google Drive (survives session timeout)\n  - resume_from_checkpoint() for Colab runtime disconnects\n\nInput is a feature DataFrame (see features.py). Rows are split into the four\nmodel branches by column name.\n"""\nfrom __future__ import annotations\n\nimport os\nimport time\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom torch.utils.data import Dataset, DataLoader\n\nfrom config import (\n    SEED, FEATURE_NAMES, N_AU, N_REGION, N_POSE, N_TEMPORAL,\n    N_EMOTION, N_SUBTYPE, EMOTION_CLASSES, SUBTYPE_CLASSES,\n    DRIVE_CHECKPOINT_DIR, LOCAL_CHECKPOINT_DIR,\n)\n\n\n# ---------------------------------------------------------------------------\n# Dataset\n# ---------------------------------------------------------------------------\nclass AffectFeatureDataset(Dataset):\n    def __init__(self, df, emotion_col="emotion", subtype_col="subtype",\n                 emo_to_idx=None, sub_to_idx=None):\n        self.df = df.reset_index(drop=True)\n        self.emo_to_idx = emo_to_idx or {c: i for i, c in enumerate(EMOTION_CLASSES)}\n        self.sub_to_idx = sub_to_idx or {c: i for i, c in enumerate(SUBTYPE_CLASSES)}\n\n    def __len__(self):\n        return len(self.df)\n\n    def _slice(self, row):\n        au = row[[f"au_{a}" for a in _AU_COLS()]].to_numpy(dtype=np.float32)\n        region = row[[f"region_{r}" for r in ["eye", "mouth", "upper_face", "lower_face"]]].to_numpy(dtype=np.float32)\n        pose = row[["pose_yaw", "pose_pitch", "pose_roll"]].to_numpy(dtype=np.float32)\n        temporal = row[[f"temp_{t}" for t in ["dwell_time", "transition_rate", "entropy", "au_variability"]]].to_numpy(dtype=np.float32)\n        return au, region, pose, temporal\n\n    def __getitem__(self, idx):\n        row = self.df.iloc[idx]\n        au, region, pose, temporal = self._slice(row)\n        emo = self.emo_to_idx[row["emotion"]]\n        sub = self.sub_to_idx[row["subtype"]]\n        return (torch.from_numpy(au), torch.from_numpy(region),\n                torch.from_numpy(pose), torch.from_numpy(temporal),\n                torch.tensor(emo, dtype=torch.long),\n                torch.tensor(sub, dtype=torch.long))\n\n\ndef _AU_COLS():\n    # keep in sync with config.AU_NAMES without circular import at module load\n    return ["AU01", "AU02", "AU04", "AU05", "AU06", "AU07", "AU09", "AU10",\n            "AU12", "AU14", "AU15", "AU17", "AU20", "AU23", "AU25", "AU26", "AU45"]\n\n\n# ---------------------------------------------------------------------------\n# Class weights\n# ---------------------------------------------------------------------------\ndef compute_class_weights(labels: list[int], n_classes: int) -> torch.Tensor:\n    counts = np.bincount(labels, minlength=n_classes).astype(np.float32)\n    counts = np.where(counts == 0, 1.0, counts)  # avoid div-by-zero\n    weights = counts.sum() / (n_classes * counts)\n    return torch.tensor(weights, dtype=torch.float32)\n\n\n# ---------------------------------------------------------------------------\n# Loss\n# ---------------------------------------------------------------------------\nclass MultiTaskLoss(nn.Module):\n    def __init__(self, emo_weight, sub_weight, lambda_subtype=0.5):\n        super().__init__()\n        self.emo = nn.CrossEntropyLoss(weight=emo_weight)\n        self.sub = nn.CrossEntropyLoss(weight=sub_weight)\n        self.lambda_subtype = lambda_subtype\n\n    def forward(self, emo_logits, sub_logits, emo_y, sub_y):\n        return self.emo(emo_logits, emo_y) + self.lambda_subtype * self.sub(sub_logits, sub_y)\n\n\n# ---------------------------------------------------------------------------\n# Checkpoint save / resume\n# ---------------------------------------------------------------------------\ndef _checkpoint_dir(use_drive: bool) -> str:\n    d = DRIVE_CHECKPOINT_DIR if use_drive else LOCAL_CHECKPOINT_DIR\n    os.makedirs(d, exist_ok=True)\n    return d\n\n\ndef save_checkpoint(path, model, optimizer, scaler, epoch, best_val_loss,\n                    emo_weight, sub_weight, extra=None):\n    torch.save({\n        "model_state": model.state_dict(),\n        "optimizer_state": optimizer.state_dict(),\n        "scaler_state": scaler.state_dict(),\n        "epoch": epoch,\n        "best_val_loss": best_val_loss,\n        "emo_weight": emo_weight,\n        "sub_weight": sub_weight,\n        "extra": extra or {},\n    }, path)\n    print(f"[checkpoint] saved -> {path}")\n\n\ndef resume_from_checkpoint(path, model, optimizer, scaler):\n    """Restore model/optimizer/scaler state and return training metadata."""\n    if not os.path.exists(path):\n        raise FileNotFoundError(f"No checkpoint at {path}")\n    ckpt = torch.load(path, map_location="cpu")\n    model.load_state_dict(ckpt["model_state"])\n    optimizer.load_state_dict(ckpt["optimizer_state"])\n    scaler.load_state_dict(ckpt["scaler_state"])\n    print(f"[resume] restored epoch {ckpt[\'epoch\']} "\n          f"(best_val_loss={ckpt[\'best_val_loss\']:.5f})")\n    return ckpt\n\n\n# ---------------------------------------------------------------------------\n# Training loop\n# ---------------------------------------------------------------------------\ndef train_one_epoch(model, loader, loss_fn, optimizer, scaler, device,\n                    use_amp=True):\n    model.train()\n    total, emo_loss_sum, sub_loss_sum = 0.0, 0.0, 0.0\n    for batch in loader:\n        au, region, pose, temporal, emo_y, sub_y = [b.to(device) for b in batch]\n        optimizer.zero_grad(set_to_none=True)\n        if use_amp:\n            with torch.cuda.amp.autocast():\n                emo_logits, sub_logits = model(au, region, pose, temporal)\n                loss = loss_fn(emo_logits, sub_logits, emo_y, sub_y)\n            scaler.scale(loss).backward()\n            scaler.step(optimizer)\n            scaler.update()\n        else:\n            emo_logits, sub_logits = model(au, region, pose, temporal)\n            loss = loss_fn(emo_logits, sub_logits, emo_y, sub_y)\n            loss.backward()\n            optimizer.step()\n        total += loss.item() * au.size(0)\n        emo_loss_sum += loss_fn.emo(emo_logits.detach(), emo_y).item() * au.size(0)\n        sub_loss_sum += loss_fn.sub(sub_logits.detach(), sub_y).item() * au.size(0)\n    n = len(loader.dataset)\n    return total / n, emo_loss_sum / n, sub_loss_sum / n\n\n\n@torch.no_grad()\ndef evaluate_loader(model, loader, loss_fn, device):\n    model.eval()\n    total, correct_emo, correct_sub = 0.0, 0, 0\n    for batch in loader:\n        au, region, pose, temporal, emo_y, sub_y = [b.to(device) for b in batch]\n        emo_logits, sub_logits = model(au, region, pose, temporal)\n        loss = loss_fn(emo_logits, sub_logits, emo_y, sub_y)\n        total += loss.item() * au.size(0)\n        correct_emo += (emo_logits.argmax(1) == emo_y).sum().item()\n        correct_sub += (sub_logits.argmax(1) == sub_y).sum().item()\n    n = len(loader.dataset)\n    return total / n, correct_emo / n, correct_sub / n\n\n\ndef train(\n    model, train_df, val_df, device,\n    epochs=50, batch_size=256, lr=1e-3, weight_decay=1e-4,\n    lambda_subtype=0.5, patience=10, use_amp=True,\n    use_drive=True, checkpoint_name="best.pt",\n    seed=SEED,\n):\n    """Full training loop with early stopping + Drive checkpointing.\n\n    Returns (model, history dict). The best checkpoint (by val loss) is written\n    to Drive (or local) as `checkpoint_name`.\n    """\n    from config import seed_everything\n    seed_everything(seed)\n\n    train_ds = AffectFeatureDataset(train_df)\n    val_ds = AffectFeatureDataset(val_df)\n    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,\n                              num_workers=2, pin_memory=True, drop_last=False)\n    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,\n                            num_workers=2, pin_memory=True)\n\n    emo_weight = compute_class_weights(\n        [train_ds.emo_to_idx[l] for l in train_df["emotion"]], N_EMOTION).to(device)\n    sub_weight = compute_class_weights(\n        [train_ds.sub_to_idx[l] for l in train_df["subtype"]], N_SUBTYPE).to(device)\n\n    loss_fn = MultiTaskLoss(emo_weight, sub_weight, lambda_subtype)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)\n    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)\n\n    ckpt_dir = _checkpoint_dir(use_drive)\n    ckpt_path = os.path.join(ckpt_dir, checkpoint_name)\n\n    best_val_loss = float("inf")\n    best_epoch = -1\n    patience_counter = 0\n    history = {"train_loss": [], "val_loss": [], "val_emo_acc": [], "val_sub_acc": []}\n\n    for epoch in range(epochs):\n        t0 = time.time()\n        tr_loss, tr_emo, tr_sub = train_one_epoch(\n            model, train_loader, loss_fn, optimizer, scaler, device, use_amp)\n        val_loss, val_emo, val_sub = evaluate_loader(model, val_loader, loss_fn, device)\n        scheduler.step()\n\n        history["train_loss"].append(tr_loss)\n        history["val_loss"].append(val_loss)\n        history["val_emo_acc"].append(val_emo)\n        history["val_sub_acc"].append(val_sub)\n\n        print(f"epoch {epoch+1:03d}/{epochs} | "\n              f"tr_loss {tr_loss:.4f} | val_loss {val_loss:.4f} | "\n              f"val_emo_acc {val_emo:.4f} | val_sub_acc {val_sub:.4f} | "\n              f"{time.time()-t0:.1f}s")\n\n        if val_loss < best_val_loss - 1e-6:\n            best_val_loss = val_loss\n            best_epoch = epoch\n            patience_counter = 0\n            save_checkpoint(ckpt_path, model, optimizer, scaler, epoch,\n                            best_val_loss, emo_weight, sub_weight,\n                            extra={"history": history})\n        else:\n            patience_counter += 1\n            print(f"  (no improvement, patience {patience_counter}/{patience})")\n            if patience_counter >= patience:\n                print(f"[early-stop] stopping at epoch {epoch+1} "\n                      f"(best epoch {best_epoch+1}, val_loss {best_val_loss:.4f})")\n                break\n\n    return model, history, ckpt_path\n'
SRC['explain.py'] = '"""\nexplain.py — SHAP and Grad-CAM explainability for the paper.\n\nSHAP   : which AU / region / head-pose / temporal features drive each\n         PTSD-subtype prediction. Output is a Fathom-style horizontal bar chart.\nGrad-CAM: overlay class-activation maps on face images to show which facial\n         region the auxiliary image pathway attends to per emotion class.\n\nBoth run on Colab T4. SHAP uses a small background set to keep runtime low;\nGrad-CAM uses the lightweight CNN backbone from model.py.\n"""\nfrom __future__ import annotations\n\nimport os\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\n\nfrom config import FEATURE_NAMES, EMOTION_CLASSES, SUBTYPE_CLASSES\n\nNAVY = "#1F3A5F"\nGRAY = "#8A919C"\nHIGHLIGHT = "#C75B39"\nFACE = "#F7F6F3"\n\n\n# ---------------------------------------------------------------------------\n# SHAP\n# ---------------------------------------------------------------------------\nclass _SHAPWrapper(nn.Module):\n    """Wraps the multi-input model so SHAP sees one flat (N, 28) input and one\n    output (subtype probabilities)."""\n\n    def __init__(self, model, output_head="subtype"):\n        super().__init__()\n        self.model = model\n        self.output_head = output_head\n\n    def forward(self, x_flat):\n        emo, sub = self.model.forward_flat(x_flat)\n        logits = sub if self.output_head == "subtype" else emo\n        return torch.softmax(logits, dim=1)\n\n\ndef shap_feature_importance(model, background_X: np.ndarray, sample_X: np.ndarray,\n                            feature_names: list[str] | None = None,\n                            output_head: str = "subtype",\n                            class_names: list[str] | None = None,\n                            out_dir: str = "/content/figures",\n                            n_background: int = 100) -> dict:\n    """Compute SHAP values and save one bar chart per class.\n\n    Returns a dict: {class_name: path_to_bar_chart}.\n    """\n    import shap\n    feature_names = feature_names or FEATURE_NAMES\n    class_names = class_names or (SUBTYPE_CLASSES if output_head == "subtype"\n                                  else EMOTION_CLASSES)\n    os.makedirs(out_dir, exist_ok=True)\n\n    model.eval()\n    wrapper = _SHAPWrapper(model, output_head)\n\n    bg = torch.tensor(background_X[:n_background], dtype=torch.float32)\n    sample = torch.tensor(sample_X[:n_background], dtype=torch.float32)\n\n    explainer = shap.GradientExplainer(wrapper, bg)\n    shap_values = explainer.shap_values(sample)\n\n    # shap_values shape: (n_classes, n_samples, n_features)\n    paths = {}\n    for ci, cls_name in enumerate(class_names):\n        vals = shap_values[ci] if isinstance(shap_values, list) else shap_values\n        mean_abs = np.abs(vals).mean(axis=0)\n        order = np.argsort(mean_abs)[::-1][:15]  # top-15 features\n        top_names = [feature_names[i] for i in order]\n        top_vals = mean_abs[order]\n\n        fig, ax = plt.subplots(figsize=(6.4, 5.0))\n        y = np.arange(len(top_names))\n        ax.barh(y, top_vals, color=NAVY, height=0.6)\n        ax.set_yticks(y)\n        ax.set_yticklabels(top_names, fontsize=9)\n        ax.invert_yaxis()\n        ax.set_xlabel("Mean |SHAP value|")\n        ax.set_title(f"{cls_name} — feature importance", fontsize=12,\n                     fontweight="bold", color="#222")\n        ax.spines["top"].set_visible(False)\n        ax.spines["right"].set_visible(False)\n        ax.grid(axis="x", color="#ECEFF3", linewidth=0.6)\n        fig.tight_layout()\n        path = os.path.join(out_dir, f"shap_{cls_name.replace(\' \', \'_\').replace(\'/\', \'_\')}.png")\n        fig.savefig(path, dpi=300)\n        plt.close(fig)\n        paths[cls_name] = path\n\n    return paths\n\n\n# ---------------------------------------------------------------------------\n# Grad-CAM\n# ---------------------------------------------------------------------------\ndef grad_cam(backbone, image_tensor, target_class: int, device="cpu"):\n    """Compute a Grad-CAM heatmap (numpy, HxW) for `target_class`."""\n    backbone.eval()\n    image_tensor = image_tensor.to(device).unsqueeze(0)\n    image_tensor.requires_grad_(True)\n\n    out = backbone(image_tensor)\n    score = out[0, target_class]\n    backbone.zero_grad()\n    score.backward()\n\n    grads = backbone.gradients  # (1, C, H\', W\')\n    acts = backbone.activations  # (1, C, H\', W\')\n    weights = grads.mean(dim=(2, 3), keepdim=True)  # global-average-pool grad\n    cam = (weights * acts).sum(dim=1, keepdim=True)\n    cam = torch.relu(cam)\n    cam = torch.nn.functional.interpolate(cam, size=image_tensor.shape[-2:],\n                                          mode="bilinear", align_corners=False)\n    cam = cam.squeeze().detach().cpu().numpy()\n    if cam.max() > 0:\n        cam = cam / cam.max()\n    return cam\n\n\ndef overlay_gradcam(image_np: np.ndarray, cam: np.ndarray, alpha: float = 0.45):\n    """Overlay a heatmap on an (H, W, 3) uint8 image using matplotlib."""\n    import matplotlib.cm as cm\n    if image_np.dtype != np.uint8:\n        image_np = (image_np * 255).astype(np.uint8)\n    fig, ax = plt.subplots(figsize=(4.2, 4.2))\n    ax.imshow(image_np)\n    ax.imshow(cam, cmap=cm.jet, alpha=alpha)\n    ax.axis("off")\n    fig.tight_layout()\n    return fig\n\n\ndef save_gradcam_grid(backbone, images: list[np.ndarray], class_names: list[str],\n                      out_path: str, device="cpu", target_per_image: list[int] | None = None):\n    """Render a grid of face images with Grad-CAM overlays per class."""\n    n = len(images)\n    cols = min(n, 4)\n    rows = int(np.ceil(n / cols))\n    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.0, rows * 3.0))\n    if rows * cols == 1:\n        axes = np.array([[axes]])\n    axes = np.atleast_2d(axes)\n\n    for i, img in enumerate(images):\n        r, c = divmod(i, cols)\n        ax = axes[r][c]\n        tc = target_per_image[i] if target_per_image else i % len(class_names)\n        img_t = torch.tensor(img.transpose(2, 0, 1), dtype=torch.float32) / 255.0\n        cam = grad_cam(backbone, img_t, tc, device)\n        ax.imshow(img)\n        ax.imshow(cam, cmap="jet", alpha=0.45)\n        ax.set_title(class_names[tc], fontsize=9)\n        ax.axis("off")\n\n    for j in range(n, rows * cols):\n        r, c = divmod(j, cols)\n        axes[r][c].axis("off")\n\n    fig.suptitle("Grad-CAM: attended facial regions per emotion class",\n                 fontsize=12, fontweight="bold")\n    fig.tight_layout()\n    fig.savefig(out_path, dpi=300)\n    plt.close(fig)\n    return out_path\n'
SRC['data_utils.py'] = '"""\ndata_utils.py — Colab dataset acquisition, structuring and integrity checks.\n\nStrategies: Kaggle API / kagglehub, gdown, and direct URL. Includes helpers to\nunzip, organise into train/val/test folders, and display a sample grid of face\nimages with their labels (and AU labels where available) as an integrity check.\n\nNOTE: DISFA and BP4D are the gold-standard AU datasets but are *request-gated*\n(no public auto-download). This module documents the request workflow and only\nauto-downloads the freely accessible top-3: CK+, AffectNet, RAF-DB.\n"""\nfrom __future__ import annotations\n\nimport os\nimport zipfile\nimport shutil\nimport numpy as np\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\n\n\n# ---------------------------------------------------------------------------\n# Low-level download helpers\n# ---------------------------------------------------------------------------\ndef download_kaggle(dataset_slug: str, out_dir: str, use_kagglehub: bool = True) -> str:\n    """Download a Kaggle dataset. Prefers kagglehub (no manual key upload)."""\n    os.makedirs(out_dir, exist_ok=True)\n    if use_kagglehub:\n        import kagglehub\n        path = kagglehub.dataset_download(dataset_slug)\n        return path\n    else:\n        # requires ~/.kaggle/kaggle.json\n        from kaggle.api.kaggle_api_extended import KaggleApi\n        api = KaggleApi()\n        api.authenticate()\n        api.dataset_download_files(dataset_slug, path=out_dir, unzip=True)\n        return out_dir\n\n\ndef download_gdown(file_id: str, out_path: str) -> str:\n    """Download a file from Google Drive by file id."""\n    import gdown\n    gdown.download(id=file_id, output=out_path, quiet=False)\n    return out_path\n\n\ndef download_url(url: str, out_path: str) -> str:\n    """Download a file from a direct URL with a progress bar."""\n    import urllib.request\n    os.makedirs(os.path.dirname(out_path), exist_ok=True)\n    urllib.request.urlretrieve(url, out_path)\n    return out_path\n\n\ndef unzip_all(src_zip: str, dest_dir: str) -> str:\n    """Extract a zip archive into dest_dir."""\n    os.makedirs(dest_dir, exist_ok=True)\n    with zipfile.ZipFile(src_zip, "r") as z:\n        z.extractall(dest_dir)\n    print(f"[unzip] {src_zip} -> {dest_dir} ({len(z.namelist())} entries)")\n    return dest_dir\n\n\n# ---------------------------------------------------------------------------\n# Train / val / test split\n# ---------------------------------------------------------------------------\ndef split_image_folders(class_root: str, out_root: str,\n                        val_ratio: float = 0.15, test_ratio: float = 0.15,\n                        seed: int = 42, copy: bool = True):\n    """Reorganise class subfolders into train/val/test splits by subject-agnostic\n    random sampling (subject-level split must be added by the study for clinical\n    datasets to avoid leakage)."""\n    from sklearn.model_selection import train_test_split\n    os.makedirs(out_root, exist_ok=True)\n    classes = sorted(d for d in os.listdir(class_root)\n                     if os.path.isdir(os.path.join(class_root, d)))\n\n    for split in ("train", "val", "test"):\n        for c in classes:\n            os.makedirs(os.path.join(out_root, split, c), exist_ok=True)\n\n    counts = {"train": 0, "val": 0, "test": 0}\n    for c in classes:\n        src = os.path.join(class_root, c)\n        files = sorted(os.listdir(src))\n        tr, rest = train_test_split(files, test_size=val_ratio + test_ratio,\n                                    random_state=seed)\n        va, te = train_test_split(rest, test_size=test_ratio / (val_ratio + test_ratio),\n                                  random_state=seed)\n        for f, split in [(f, "train") for f in tr] + [(f, "val") for f in va] + [(f, "test") for f in te]:\n            op = shutil.copy2 if copy else shutil.move\n            op(os.path.join(src, f), os.path.join(out_root, split, c, f))\n            counts[split] += 1\n    print("[split] train/val/test counts:", counts)\n    return out_root\n\n\n# ---------------------------------------------------------------------------\n# Integrity check: display sample images + labels\n# ---------------------------------------------------------------------------\ndef display_sample_grid(image_paths: list[str], labels: list[str],\n                        au_labels: list[str] | None = None,\n                        title: str = "Dataset integrity check",\n                        n_rows: int = 2, n_cols: int = 4):\n    """Show a grid of face images with their emotion (and optional AU) labels."""\n    from PIL import Image\n    n = min(len(image_paths), n_rows * n_cols)\n    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.6, n_rows * 2.8))\n    axes = np.atleast_2d(axes)\n    for i in range(n):\n        r, c = divmod(i, n_cols)\n        img = np.asarray(Image.open(image_paths[i]).convert("RGB"))\n        axes[r][c].imshow(img)\n        cap = labels[i]\n        if au_labels is not None and au_labels[i]:\n            cap += f"\\nAU: {au_labels[i]}"\n        axes[r][c].set_title(cap, fontsize=8)\n        axes[r][c].axis("off")\n    for j in range(n, n_rows * n_cols):\n        r, c = divmod(j, n_cols)\n        axes[r][c].axis("off")\n    fig.suptitle(title, fontsize=12, fontweight="bold")\n    fig.tight_layout()\n    plt.show()\n    return fig\n\n\n# ---------------------------------------------------------------------------\n# Request-form workflow (DISFA / BP4D)\n# ---------------------------------------------------------------------------\nREQUEST_GATED = {\n    "DISFA": {\n        "url": "https://mohammadmahoor.com/pages/databases/disfa/",\n        "note": "Email request to the University of Denver team; free for academic "\n                "use under a signed agreement. 27 subjects, 12 AUs @ 0-5 intensity, "\n                "spontaneous (20 fps video). Gold standard for spontaneous AU intensity.",\n    },\n    "BP4D": {\n        "url": "https://binghamton.technologypublisher.com/tech/BP4D",\n        "note": "Request via Binghamton University technology transfer; free academic. "\n                "41 subjects (BP4D+) / 140 (BP4D+), 2D+3D video, FACS-coded AUs with "\n                "intensity. Best AU + video for dynamic affect.",\n    },\n}\n\n\ndef print_request_instructions(dataset: str) -> None:\n    """Print the manual request workflow for a request-gated dataset."""\n    info = REQUEST_GATED[dataset]\n    print(f"\\n=== {dataset} (request-gated, not auto-downloadable) ===\\n"\n          f"URL : {info[\'url\']}\\n"\n          f"How : {info[\'note\']}\\n"\n          f"Once received, place the data under /content/data/{dataset.lower()}/ "\n          f"and run the feature pipeline as usual.\\n")\n'

for _name, _text in SRC.items():
    with open(os.path.join(PIPE, _name), 'w') as _f:
        _f.write(_text)
print('wrote', len(SRC), 'modules to', PIPE)

import config
from config import seed_everything, SEED, FEATURE_NAMES, EMOTION_CLASSES, SUBTYPE_CLASSES
from features import build_feature_frame, generate_synthetic_dataset
from tables import save_tables, emotion_table_markdown, subtype_table_markdown
from evaluate import run_evaluation, per_class_metrics, attach_auc
from model import build_model, MultiInputPTSDAffectModel, GradCAMBackbone
from train import train, resume_from_checkpoint, save_checkpoint, AffectFeatureDataset
from explain import shap_feature_importance, grad_cam, save_gradcam_grid
import data_utils as du

seed_everything(SEED)
print('pipeline modules loaded; seed =', SEED)


In [ ]:
# ============================================================
# Smoke test — tiny end-to-end run on SYNTHETIC data (run first!)
# Verifies the whole pipeline wiring before any real download.
# ============================================================
import torch
import numpy as np

df, X, y_emo, y_sub = generate_synthetic_dataset(n_sequences=6, frames_per_seq=60, seed=0)
print('Feature frame:', df.shape, '| feature dim:', X.shape[1], '(expect 28)')
assert X.shape[1] == len(FEATURE_NAMES) == 28, 'feature dimension mismatch'
assert df['temp_entropy'].notna().all(), 'temporal features contain NaN'

# Build + forward
model = build_model()
model.eval()
au = torch.randn(8, 17); rg = torch.randn(8, 4); ps = torch.randn(8, 3); tm = torch.randn(8, 4)
emo_logits, sub_logits = model(au, rg, ps, tm)
print('emotion logits:', tuple(emo_logits.shape), '| subtype logits:', tuple(sub_logits.shape))
assert emo_logits.shape[1] == 6 and sub_logits.shape[1] == 3
print(model.summary())
print('\nSMOKE TEST PASSED ✅')


## 1. Data

Acquire the **top-3 freely downloadable** datasets — CK+, AffectNet, RAF-DB —
and document the request workflow for the request-gated gold standards
(DISFA, BP4D). Full ranking + rationale: see `DATASET_REPORT.md` / `MANIFEST.md`.


In [ ]:
# ============================================================
# 1. Data acquisition  (Kaggle / gdown / direct URL)
# ============================================================
# --- Strategy A: Kaggle datasets (no manual key with kagglehub) ---
# CK+ (Extended Cohn-Kanade) — AU + emotion labels, small, ideal Colab starter
ck_path = du.download_kaggle('sharmaroshan/extended-cohn-kanade', '/content/data/ck+')

# AffectNet (8-emotion subset)
affect_path = du.download_kaggle('noamsegal/affectnet-training-data', '/content/data/affectnet')

# RAF-DB (single-label subset)
raf_path = du.download_kaggle('shuvoalok/raf-db-dataset', '/content/data/rafdb')

# --- Strategy B: direct URL / gdown (fallback) ---
# ck_zip = du.download_gdown('<FILE_ID>', '/content/data/ck+.zip')
# du.unzip_all(ck_zip, '/content/data/ck+_unzipped')

# --- Structure into train/val/test (class folders) ---
# du.split_image_folders('/content/data/ck+/CK+48', '/content/data/ck+_split',
#                        val_ratio=0.15, test_ratio=0.15, seed=SEED)

# --- Integrity check: show sample faces + labels ---
# du.display_sample_grid(image_paths, labels, au_labels=None,
#                        title='CK+ sample', n_rows=2, n_cols=4)

# --- Request-gated gold standards (manual, see DATASET_REPORT.md) ---
du.print_request_instructions('DISFA')
du.print_request_instructions('BP4D')
print('\nData section ready. Set real paths once downloaded.')


## 2. Features

Extract, per face frame: **17 AU intensities**, **4 region-weighted scores**
(eye / mouth / upper / lower face), **3 head-pose angles** (yaw/pitch/roll), and
**4 temporal features** (dwell time, transition rate, entropy, AU variability)
→ a **28-dim vector** per frame.


In [ ]:
# ============================================================
# 2. Feature extraction  (AU + region + pose + temporal)
# ============================================================
# Real pipeline: get AU + pose from py-feat (or OpenFace) per frame.
#   from feat import Detector
#   detector = Detector(au_model='rf', emotion_model='resmasknet')
#   preds = detector.detect_video('/path/to/video.mp4')
#   au = preds.aus()      # (n_frames, 20) -> subset to the 17 canonical AUs
#   pose = preds.pose()   # (n_frames, 6) -> yaw/pitch/roll
#
# Here we demonstrate the full frame-DataFrame builder on synthetic data.

df, X, y_emo, y_sub = generate_synthetic_dataset(n_sequences=6, frames_per_seq=60, seed=0)

# The 28 feature columns in order
print('Feature columns (28):')
for i, c in enumerate(FEATURE_NAMES):
    print(f'  {i:2d} {c}')

# Inspect a few rows
print(df[['sequence_id', 'frame_idx', 'au_AU04', 'au_AU12', 'region_mouth',
          'region_upper_face', 'pose_yaw', 'temp_dwell_time',
          'temp_transition_rate', 'temp_entropy', 'temp_au_variability',
          'emotion', 'subtype']].head(8).to_string(index=False))

# Save frame-level features
os.makedirs('/content/data/features', exist_ok=True)
df.to_csv('/content/data/features/frame_features.csv', index=False)
print('\nSaved /content/data/features/frame_features.csv')

# Visualise the feature structure (one sample sequence)
import matplotlib.pyplot as plt
seq = df[df.sequence_id == 0]
fig, axs = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
axs[0].plot(seq.frame_idx, seq[['au_AU04', 'au_AU06', 'au_AU12']])
axs[0].set_ylabel('AU intensity'); axs[0].legend(['AU04 brow', 'AU06 cheek', 'AU12 lip'], fontsize=8, frameon=False)
axs[1].plot(seq.frame_idx, seq[['region_eye', 'region_mouth', 'region_upper_face', 'region_lower_face']])
axs[1].set_ylabel('Region score'); axs[1].legend(fontsize=8, frameon=False)
axs[2].plot(seq.frame_idx, seq[['temp_dwell_time', 'temp_transition_rate', 'temp_entropy', 'temp_au_variability']])
axs[2].set_ylabel('Temporal'); axs[2].set_xlabel('frame'); axs[2].legend(fontsize=8, frameon=False)
plt.tight_layout(); plt.show()


## 3. Model

**MultiInputPTSDAffectModel** — four branch encoders (Linear → BatchNorm → ReLU
→ Dropout) → late-fusion concatenation → shared MLP → two heads:

1. **Emotion head (6):** Fear, Anger, Sadness, Neutral, Surprise, Flat/Blunted Affect
2. **Subtype head (3):** Classic PTSD, D-PTSD, Control


In [ ]:
# ============================================================
# 3. Model — multi-input late-fusion classifier
# ============================================================
model = build_model(use_gradient_checkpointing=True)  # toggle to save VRAM
print(model.summary())

# Sample forward pass (shapes only)
import torch
b = 8
emo_logits, sub_logits = model(
    torch.randn(b, 17), torch.randn(b, 4), torch.randn(b, 3), torch.randn(b, 4))
print('emotion logits:', tuple(emo_logits.shape), '| subtype logits:', tuple(sub_logits.shape))

# Auxiliary CNN used only for Grad-CAM visualisation
backbone = GradCAMBackbone()
print('GradCAM backbone params:',
      sum(p.numel() for p in backbone.parameters()))


## 4. Train

Colab-optimised loop: **mixed precision (AMP)**, **gradient checkpointing**,
**class-weighted loss** (both heads), **early stopping (patience 10)**, and
**Google Drive checkpointing** with resume-from-checkpoint for session timeouts.


In [ ]:
# ============================================================
# 4. Training  (AMP + early stopping + Drive checkpoint / resume)
# ============================================================
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

# Load frame features (real run) — or use synthetic for the demo
# import pandas as pd
# df = pd.read_csv('/content/data/features/frame_features.csv')
df, X, y_emo, y_sub = generate_synthetic_dataset(n_sequences=12, frames_per_seq=80, seed=SEED)

# Simple sequential split (use subject-level split for the real study to avoid leakage)
split = int(0.8 * df.sequence_id.nunique())
tr_mask = df.sequence_id < split
train_df, val_df = df[tr_mask].copy(), df[~tr_mask].copy()
print(f'train frames {len(train_df)} | val frames {len(val_df)}')

model = build_model(use_gradient_checkpointing=True).to(device)

# Train (saves best checkpoint to Drive; resume happens automatically if it exists)
model, history, ckpt_path = train(
    model, train_df, val_df, device,
    epochs=3,               # raise for the real run (e.g. 50)
    batch_size=256, lr=1e-3, weight_decay=1e-4,
    lambda_subtype=0.5, patience=10,
    use_amp=(device.type == 'cuda'),
    use_drive=True, checkpoint_name='best.pt',
    seed=SEED,
)

# --- Resume after a Colab session timeout (demonstration) ---
# import torch
# model2 = build_model().to(device)
# opt2 = torch.optim.AdamW(model2.parameters(), lr=1e-3)
# scaler2 = torch.cuda.amp.GradScaler(enabled=True)
# meta = resume_from_checkpoint('/content/drive/MyDrive/ptsd_affect/checkpoints/best.pt',
#                               model2, opt2, scaler2)
# print('resumed from epoch', meta['epoch'])


## 5. Evaluate

Per-class precision / recall / F1 for six emotions, a confusion matrix with
**fear–surprise** and **anger–disgust** pairs highlighted, **ROC curves**, the
**PTSD-subtype report** (Classic / D-PTSD / Control), and paper-ready summary
tables (Markdown + LaTeX).


In [ ]:
# ============================================================
# 5. Evaluation + figures + paper tables
# ============================================================
import numpy as np

# --- Run the model on the val split to get predictions / probabilities ---
# (demo uses synthetic labels; replace with real model outputs)
from evaluate import EMOTION_CLASSES as _EC
rng = np.random.default_rng(0)
y_emo_true = list(val_df['emotion'])
y_emo_pred = [e if rng.random() < 0.7 else _EC[rng.integers(6)] for e in y_emo_true]
y_emo_score = np.zeros((len(y_emo_true), 6))
for i, e in enumerate(y_emo_true):
    j = _EC.index(e); y_emo_score[i, j] = rng.uniform(0.5, 1.0)
y_emo_score = (y_emo_score + 0.05) / (y_emo_score + 0.05).sum(1, keepdims=True)

y_sub_true = list(val_df['subtype'])
y_sub_pred = [s if rng.random() < 0.6 else SUBTYPE_CLASSES[rng.integers(3)] for s in y_sub_true]

os.makedirs('/content/figures', exist_ok=True)
bundle = run_evaluation(y_emo_true, y_emo_pred, y_emo_score, y_sub_true, y_sub_pred,
                        '/content/figures')

# --- Paper-ready tables ---
emo_rows = [{'class': c, **{k: bundle['emotion_metrics'][c][k]
                            for k in ('precision', 'recall', 'f1', 'auc')}}
            for c in EMOTION_CLASSES]
tpaths = save_tables(emo_rows, bundle['subtype_metrics'], '/content/figures',
                     classes=EMOTION_CLASSES)

print('\n===== Emotion metrics (Markdown) =====')
print(tpaths['emotion_markdown'])
print('\n===== Subtype metrics (Markdown) =====')
print(tpaths['subtype_markdown'])
print('\n===== Subtype report (sklearn) =====')
print(bundle['subtype_metrics']['report'])
print('\nFigures:', bundle['figures'])
print('Tables :', {k: v for k, v in tpaths.items() if k.endswith(('_md', '_tex'))})


## 6. Explain

- **SHAP** — which AU / region / pose / temporal features drive each subtype
  prediction (Fathom-style bar charts for the Results section).
- **Grad-CAM** — which facial regions the auxiliary image pathway attends to,
  per emotion class.


In [ ]:
# ============================================================
# 6. Explainability  (SHAP + Grad-CAM)
# ============================================================
import numpy as np, torch

# --- SHAP: feature importance per PTSD subtype ---
# Use a small background/sample set (T4-friendly). Replace X with real features.
_, X_bg, _, _ = generate_synthetic_dataset(n_sequences=4, frames_per_seq=40, seed=7)
shap_paths = shap_feature_importance(
    model.to('cpu'), X_bg, X_bg,
    feature_names=FEATURE_NAMES, output_head='subtype',
    class_names=SUBTYPE_CLASSES, out_dir='/content/figures', n_background=80)
print('SHAP charts:', shap_paths)

# --- Grad-CAM: attended facial regions per emotion ---
# Replace `demo_faces` with real face crops (e.g. from AffectNet / CK+).
demo_faces = [np.random.randint(0, 255, (112, 112, 3), dtype=np.uint8)
              for _ in range(6)]
grid_path = save_gradcam_grid(GradCAMBackbone(), demo_faces, EMOTION_CLASSES,
                              '/content/figures/gradcam_grid.png', device='cpu')
print('Grad-CAM grid:', grid_path)
print('\nExplain section complete.')
